<a href="https://colab.research.google.com/github/vipulsahu0629/PW_Assessment/blob/main/React_Hooks_Effects.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# React Hooks, Effects & More | Assignment
**Assignment Code:** FSD-AG-009                     
 **Total Marks:** 200

## **Question 1:** In simple words, what are React Hooks? Create a small component that shows a button. When the button is clicked, toggle the text between "On" and "Off".

**Answer:**

**React Hooks** are special functions (like `useState`, `useEffect`, `useReducer`, `useRef`, `useContext`) that let functional components use features — state, lifecycle behavior, refs, context — that used to require class components. They let you "hook into" React's internals from a plain function, keeping components simple and reusable.

```jsx
import React, { useState } from 'react';

function ToggleButton() {
  const [isOn, setIsOn] = useState(false);

  return (
    <button onClick={() => setIsOn(!isOn)}>
      {isOn ? 'On' : 'Off'}
    </button>
  );
}

export default ToggleButton;
```

`useState(false)` creates a piece of state (`isOn`) starting at `false`. Clicking the button calls `setIsOn(!isOn)`, flipping the value and re-rendering the component with the new "On"/"Off" text.

## **Question 2:** Make a counter with a button. Whenever the count changes, log "Count changed" using `useEffect`.

**Answer:**

```jsx
import React, { useState, useEffect } from 'react';

function Counter() {
  const [count, setCount] = useState(0);

  useEffect(() => {
    console.log('Count changed');
  }, [count]);

  return (
    <div>
      <p>Count: {count}</p>
      <button onClick={() => setCount(count + 1)}>Increment</button>
    </div>
  );
}

export default Counter;
```

`useEffect` runs after every render by default. Since this effect has `[count]` as its dependency array, it only re-runs when `count` changes — so it fires once on mount, and again each time the button updates the count.

## **Question 3:** Write a component that fetches users from `https://jsonplaceholder.typicode.com/users` only once on mount and shows how many users were loaded.

**Answer:**

```jsx
import React, { useState, useEffect } from 'react';

function UserCount() {
  const [count, setCount] = useState(null);
  const [error, setError] = useState(null);

  useEffect(() => {
    fetch('https://jsonplaceholder.typicode.com/users')
      .then(response => response.json())
      .then(data => setCount(data.length))
      .catch(err => setError(err.message));
  }, []); // empty dependency array -> runs only once, on mount

  if (error) return <p>Failed to load users: {error}</p>;
  if (count === null) return <p>Loading users...</p>;

  return <p>{count} users were loaded.</p>;
}

export default UserCount;
```

Passing an **empty dependency array `[]`** to `useEffect` means the effect runs exactly once — right after the component's first render (mount) — and never again, which is the correct way to fetch data "only once".

## **Question 4:** Create a component that starts a timer with `setInterval` to increase a "seconds" counter every second. Stop the timer when the component unmounts.

**Answer:**

```jsx
import React, { useState, useEffect } from 'react';

function Timer() {
  const [seconds, setSeconds] = useState(0);

  useEffect(() => {
    const intervalId = setInterval(() => {
      setSeconds(prev => prev + 1);
    }, 1000);

    // Cleanup: runs when the component unmounts
    return () => clearInterval(intervalId);
  }, []);

  return <p>Seconds elapsed: {seconds}</p>;
}

export default Timer;
```

The `useEffect` cleanup function (the function returned from inside the effect) runs automatically when the component unmounts. Here it calls `clearInterval` to stop the timer, preventing memory leaks or state updates on an unmounted component.

## **Question 5:** Build a small form with a text input. If the input is empty, show "Please type your name". If it has text, show "Hello, <name>".

**Answer:**

```jsx
import React, { useState } from 'react';

function NameForm() {
  const [name, setName] = useState('');

  return (
    <div>
      <input
        type="text"
        value={name}
        onChange={(e) => setName(e.target.value)}
        placeholder="Type your name"
      />
      <p>{name.trim() === '' ? 'Please type your name' : `Hello, ${name}`}</p>
    </div>
  );
}

export default NameForm;
```

The input's value is stored in state (`name`) and updated on every keystroke via `onChange`, making this a **controlled input** — React state is always the single source of truth for what's displayed.

## **Question 6:** You have a theme value ("light" or "dark") that is needed by a deep child button. Show how to pass it without sending theme through every parent prop.

**Answer:**

```jsx
import React, { createContext, useContext } from 'react';

const ThemeContext = createContext('light');

function ThemedButton() {
  const theme = useContext(ThemeContext);

  return (
    <button
      style={{
        backgroundColor: theme === 'dark' ? '#333' : '#eee',
        color: theme === 'dark' ? '#fff' : '#000',
      }}
    >
      I am a {theme} themed button
    </button>
  );
}

function MiddleLayer() {
  // No need to pass "theme" as a prop here at all
  return <ThemedButton />;
}

function App() {
  return (
    <ThemeContext.Provider value="dark">
      <MiddleLayer />
    </ThemeContext.Provider>
  );
}

export default App;
```

This is exactly what the **Context API** is for. `createContext` creates a Context object; a `Provider` at the top supplies the value, and any descendant — no matter how deeply nested — can read it directly with `useContext`, skipping "prop drilling" through every intermediate component.

## **Question 7:** Make a counter with two buttons: "+1" and "-1". Use `useReducer` instead of `useState`.

**Answer:**

```jsx
import React, { useReducer } from 'react';

function reducer(state, action) {
  switch (action.type) {
    case 'increment':
      return { count: state.count + 1 };
    case 'decrement':
      return { count: state.count - 1 };
    default:
      return state;
  }
}

function ReducerCounter() {
  const [state, dispatch] = useReducer(reducer, { count: 0 });

  return (
    <div>
      <p>Count: {state.count}</p>
      <button onClick={() => dispatch({ type: 'increment' })}>+1</button>
      <button onClick={() => dispatch({ type: 'decrement' })}>-1</button>
    </div>
  );
}

export default ReducerCounter;
```

`useReducer` manages state via a **reducer function** `(state, action) => newState`, dispatched with `dispatch(action)`. It's a good fit when state updates follow a clear set of actions, as with increment/decrement here.

## **Question 8:** Given an array `nums = [1,2,3,4]` and a filter value `min`, show how to compute a filtered list only when `nums` or `min` change. Also pass a stable click handler to a child list item.

**Answer:**

```jsx
import React, { useState, useMemo, useCallback } from 'react';

const nums = [1, 2, 3, 4];

const ListItem = React.memo(function ListItem({ value, onClick }) {
  return <li onClick={() => onClick(value)}>{value}</li>;
});

function FilteredList() {
  const [min, setMin] = useState(2);
  const [lastClicked, setLastClicked] = useState(null);

  const filtered = useMemo(() => {
    console.log('Recomputing filtered list');
    return nums.filter(n => n >= min);
  }, [nums, min]);

  const handleClick = useCallback((value) => {
    setLastClicked(value);
  }, []);

  return (
    <div>
      <label>
        Min:{' '}
        <input
          type="number"
          value={min}
          onChange={(e) => setMin(Number(e.target.value))}
        />
      </label>
      <ul>
        {filtered.map(n => (
          <ListItem key={n} value={n} onClick={handleClick} />
        ))}
      </ul>
      <p>Last clicked: {lastClicked ?? 'none'}</p>
    </div>
  );
}

export default FilteredList;
```

- `useMemo` **memoizes** the filtered array, recomputing it only when `nums` or `min` change — avoiding unnecessary filtering on unrelated re-renders.
- `useCallback` **memoizes the function itself**, giving child components (like `ListItem`) a stable reference for the handler across re-renders, which helps avoid unnecessary re-renders of memoized children (`ListItem` is wrapped in `React.memo` to take advantage of this).

## **Question 9:** Create a controlled input for email. As the user types, show "Email: <value>". Why do we keep the input's value in state?

**Answer:**

```jsx
import React, { useState } from 'react';

function EmailForm() {
  const [email, setEmail] = useState('');

  return (
    <div>
      <input
        type="email"
        value={email}
        onChange={(e) => setEmail(e.target.value)}
        placeholder="Enter your email"
      />
      <p>Email: {email}</p>
    </div>
  );
}

export default EmailForm;
```

We keep the input's value in state so that **React remains the single source of truth** for the UI. This makes the current value always available to the component's logic (for validation, formatting, submission, conditional rendering, etc.), and keeps the displayed value and the internal data perfectly in sync — rather than having to read the DOM directly to know what the user typed.

## **Question 10:** Make a button that focuses an input using a ref. Return adjacent elements without extra wrapper nodes. In one sentence, say what React's update process does.

**Answer:**

```jsx
import React, { useRef } from 'react';

function FocusInput() {
  const inputRef = useRef(null);

  const handleFocus = () => {
    inputRef.current.focus();
  };

  return (
    <>
      <input ref={inputRef} type="text" placeholder="Click the button to focus me" />
      <button onClick={handleFocus}>Focus the input</button>
    </>
  );
}

export default FocusInput;
```

- `useRef` creates a mutable reference (`inputRef.current`) that points directly to the DOM node, letting us call imperative methods like `.focus()` on it.
- `React.Fragment` (shorthand `<>...</>`) lets a component return multiple adjacent elements **without** adding an extra wrapper `<div>` to the actual DOM.
- **In one sentence:** React's update process (reconciliation) compares the new Virtual DOM tree with the previous one and applies only the minimal set of changes needed to the real DOM, rather than re-rendering the whole page.